# 第 11 课 · 一篮子公司为什么同天倒霉（浏览器版）

配套课文：[docs/m0-foundations/11-correlation-and-baskets.md](../docs/m0-foundations/11-correlation-and-baskets.md)。

**怎么用**：本地装了 Jupyter 就直接打开这一页；没有环境就上 https://jupyter.org/try （JupyterLite）或 Colab，把这份 notebook 整份放进去，每一格按 `Shift + Enter` 运行。只用 Python 标准库，不需要安装任何东西。

每格跑出来的数字和课文里印的逐字一致（种子钉在 7）。


## 一、你明明买了十只，为什么那天全绿

你有三个账号，都是生活类选题。你告诉自己这叫矩阵、叫分散风险。某天平台改了推荐权重，三个账号同一天播放量腰斩。

基金也一样。你买了十只，心想"总不会一起跌吧"，结果某天打开账户，十只全绿。

**"一篮子里东西多"和"篮子里的东西会不会一起倒霉"是两件事。** 前者你数得清（十只就是十只），后者你一直没算过。这一节课就把它变成一个数，然后你会看到：分散这件事有一个消不掉的地板，而地板的高度就是那个数决定的。

第 06 课我们说过一句话——**期望能拆开相加，方差不能**。当时欠你的那一项，今天补上。


## 二、"一起动"是可以数的

手上两列数，一列是每天 A 的涨跌，一列是每天 B 的涨跌。要判断它们是不是一起动，做法只有一步：**先看 A 高于自己平均的那些天，B 是不是也高于自己的平均。**

是，就是正相关；反过来就是负相关；高低对不上就是各动各的，相关约等于 0。

先放上第 08 课那台量散布的机器 `spread`，再写 `corr`。`zip` 是第 06 课那招"两列数一次各拿一个，凑成一对"。整个函数做的事就是上面那句话：把"各自偏离自己的平均"配成对乘起来，再除以两列各自的散布——除这一步是为了把结果压进 −1 到 1 之间，好让"0.7"这种数在不同问题上可比。

（下面这一格只是定义，运行它不会打印任何东西。）


In [1]:
import math
import random


def spread(vals):
    n = len(vals)
    m = sum(vals) / n
    return math.sqrt(sum((x - m) ** 2 for x in vals) / (n - 1))


def corr(xs, ys):
    """两列数有多"一起动"：-1 到 1。"""
    n = len(xs)
    mx = sum(xs) / n
    my = sum(ys) / n
    sx = spread(xs)
    sy = spread(ys)
    return sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / (n - 1) / (sx * sy)


**这个数叫相关系数，记作 $\rho$。** 它是这门课第一个真正描述"两件事之间的关系"的数，也是原书第 2 章后半节（多维正态）和第 9 章（组合风险）共同的地基。

它灵不灵？我造几组已知答案的数据来验：`market_pair` 用的搭法是"每个账号/每只股票都受同一个大环境影响，再各加一点自己的运气"，共同的大环境占多少由 $\rho$ 控制。跑两万天：


In [2]:
def market_pair(rho, n):
    """造 n 天的两列数，让它们的相关大约是 rho：共同的"大环境"加各自的"小运气"。"""
    xs = []
    ys = []
    for _ in range(n):
        common = random.gauss(0, 1)
        xs.append(math.sqrt(rho) * common + math.sqrt(1 - rho) * random.gauss(0, 1))
        ys.append(math.sqrt(rho) * common + math.sqrt(1 - rho) * random.gauss(0, 1))
    return xs, ys


In [3]:
random.seed(7)
for target in [0.0, 0.3, 0.7, 0.9]:
    xs, ys = market_pair(target, 20000)
    print(f"设定相关 {target:.1f} → 实测 corr {corr(xs, ys):.3f}")


设定相关 0.0 → 实测 corr 0.013
设定相关 0.3 → 实测 corr 0.309
设定相关 0.7 → 实测 corr 0.699
设定相关 0.9 → 实测 corr 0.900


量尺是对的。


## 三、两个加起来，起伏不是相加

你中午点外卖、晚上打车，两笔钱合起来是一天的开销。如果它们各自动，合起来的起伏是多少？

很多人会答"两个加起来"。第 06 课已经说过这是错的，但错多少一直没给数。现在有数了。让两列起伏都是 1，各跑两万天，把每天的两个数相加再看散布：


In [4]:
random.seed(7)
for r in [0.0, 0.3, 0.7, 1.0]:
    xs, ys = market_pair(r, 20000)
    summed = [x + y for x, y in zip(xs, ys)]
    theory = math.sqrt(1 + 1 + 2 * r)
    print(f"两者相关 {r:.1f}：加起来的起伏 实测 {spread(summed):.3f} / 理论 {theory:.3f}（各自起伏都是 1.000）")


两者相关 0.0：加起来的起伏 实测 1.424 / 理论 1.414（各自起伏都是 1.000）
两者相关 0.3：加起来的起伏 实测 1.617 / 理论 1.612（各自起伏都是 1.000）
两者相关 0.7：加起来的起伏 实测 1.836 / 理论 1.844（各自起伏都是 1.000）
两者相关 1.0：加起来的起伏 实测 2.002 / 理论 2.000（各自起伏都是 1.000）


这张表值得慢慢看：

- 完全各动各的（$\rho = 0$），合起来的起伏是 $\sqrt{2} = 1.41$，**既不是 1，也不是 2**。第 08 课那个"A/B 两组比较的毛边是单组的 √2 倍"，出处就在这里。
- 完全同步（$\rho = 1$），才是老老实实的 1 + 1 = 2。
- 中间的值由 $\rho$ 决定。

写成式子。它替你回答的问题是：**"两个来源合起来，会比各自更晃多少？"**

$$\operatorname{Var}(X+Y) = \operatorname{Var}(X) + \operatorname{Var}(Y) + 2\rho\,\sigma_X\sigma_Y$$

左边是"合起来的方差"，前两项是各自的方差，第三项就是第 06 课欠你的那一项。$\rho = 0$ 时它消失，$\rho = 1$ 时它让 $1 + 1$ 变成 $4$（开根号就是 2）。

**这一项有个要命的性质：你买多少东西都消不掉它。** 下一节就说这件事。


## 四、一篮子的起伏，和它消不掉的那部分

现在把"分散"变成一个动作：$n$ 只资产各买十分之一。每只单看起伏都是 1，彼此相关 $\rho$。

最自然的搭法是"每个账号/每只股票都受同一个大环境影响，再各加一点自己的运气"：

```python
value = math.sqrt(rho) * common + math.sqrt(1 - rho) * random.gauss(0, 1)
```

`common` 那天只有一个（平台改规则、降息、疫情），每只都沾一点；后面那项是各自的私事。这样造出来的两两相关正好是 $\rho$（课文进阶框给三行验证）。

先造一台篮子机器 `basket_sd`：每天抽一个 `common`，$n$ 只各沾一份再加自己的运气，等权平均，最后看这个篮子每天的起伏（这一格也只是定义）。跑两万天，$\rho = 0.3$，看篮子每天的起伏：


In [5]:
def basket_sd(rho, n, days):
    """n 只资产等权合成一个篮子，看这个篮子每天的起伏（单只起伏按 1 算）。"""
    out = []
    for _ in range(days):
        common = random.gauss(0, 1)
        total = 0
        for _ in range(n):
            total += math.sqrt(rho) * common + math.sqrt(1 - rho) * random.gauss(0, 1)
        out.append(total / n)
    return spread(out)


In [6]:
random.seed(7)
rho = 0.3
for n in [1, 2, 10, 50]:
    measured = basket_sd(rho, n, 20000)
    theory = math.sqrt(rho + (1 - rho) / n)
    print(f"{n:>2} 只、彼此相关 {rho}：篮子起伏 实测 {measured:.3f} / 理论 {theory:.3f}（单只 = 1.000）")


 1 只、彼此相关 0.3：篮子起伏 实测 0.996 / 理论 1.000（单只 = 1.000）
 2 只、彼此相关 0.3：篮子起伏 实测 0.808 / 理论 0.806（单只 = 1.000）
10 只、彼此相关 0.3：篮子起伏 实测 0.608 / 理论 0.608（单只 = 1.000）
50 只、彼此相关 0.3：篮子起伏 实测 0.559 / 理论 0.560（单只 = 1.000）


理论那列是这一行：

$$\text{篮子的起伏} = \sqrt{\rho + \frac{1-\rho}{n}}$$

它回答的问题就是第一节那个：**"我买到第 n 只，到底降了多少风险？"** 而式子把答案摊得很直白：加进去的那部分风险是 $(1-\rho)/n$，它会被 $n$ 摊薄到没有；**留下来的 $\rho$ 摊不掉，因为它是所有资产共享的那一个大环境。**

把地板列出来（同一套式子，只是换 $\rho$ 和 $n$）：


In [7]:
for r in [0.0, 0.3, 0.7]:
    one = 1.0
    ten = math.sqrt(r + (1 - r) / 10)
    hundred = math.sqrt(r + (1 - r) / 100)
    floor = math.sqrt(r)
    print(f"相关 {r:.1f}：1 只 {one:.3f} → 10 只 {ten:.3f} → 100 只 {hundred:.3f}；地板（无穷多只）{floor:.3f}")
print("换算成每天 2% 的单只起伏：相关 0.3 时，1 只 2.00% → 10 只 1.22% → 100 只 1.11%，再怎么加也到不了 1.10% 以下。")


相关 0.0：1 只 1.000 → 10 只 0.316 → 100 只 0.100；地板（无穷多只）0.000
相关 0.3：1 只 1.000 → 10 只 0.608 → 100 只 0.554；地板（无穷多只）0.548
相关 0.7：1 只 1.000 → 10 只 0.854 → 100 只 0.838；地板（无穷多只）0.837
换算成每天 2% 的单只起伏：相关 0.3 时，1 只 2.00% → 10 只 1.22% → 100 只 1.11%，再怎么加也到不了 1.10% 以下。


上面那几行画成图就是三条往下弯、然后被同色虚线接住的曲线。橙点是我真去模拟出来的篮子起伏（相关 0.3、两万天），它们就落在式子的线上。

![篮子起伏随只数往下弯的三条曲线：相关 0、0.3、0.7；同色虚线是地板 √ρ；相关 0.3 那条从 1 只 1.000 掉到 10 只 0.608、100 只 0.554，地板 0.548](https://raw.githubusercontent.com/JavenZheng/monte-carlo-finance/main/assets/fig11_diversification.svg)

图由 [code/fig11_diversification.py](../code/fig11_diversification.py) 生成，橙点用的就是上面这一轮抽签（种子 7）。

三件事一次看清：

**从 1 只到 10 只，砍掉将近四成风险；从 10 只到 100 只，只砍掉 9%。** 你的分散绝大部分在前十只就拿到了，后面买的是手续费。

**真正决定效果的是 $\rho$，不是 $n$。** $\rho = 0.7$ 时买一百只的起伏还有 0.838，几乎是单只的 84%。**三个同题材账号不是三个账号，是一个账号发了三遍。**

**$\rho$ 变小比 $n$ 变大值钱。** 把 $\rho$ 从 0.7 降到 0.3，等价于多买很多只；而换一个赛道就能把 $\rho$ 降下来，不用花一份钱。

顺带你现在能解释第一节那两个现象了：那天全绿，不是你的十只基金出了事，是它们共享的那个 `common` 出了事。


## 五、你算出来的那个相关，本身也在飘

第四节的整套结论都建立在"你知道 $\rho$"上。可现实里你只有几十天的记录。

上一课说过分位数有毛边，这一课的 $\rho$ 一样有。下面这一段做的事有点残忍：**造两列真·毫不相干的数据（$\rho$ 设为 0），只给你八天，然后算相关系数。** 五千轮：


In [8]:
for days in [8, 30, 100]:
    random.seed(7)
    big = 0
    trials = 5000
    values = []
    for _ in range(trials):
        a, b = market_pair(0.0, days)
        c = corr(a, b)
        values.append(abs(c))
        if abs(c) > 0.5:
            big += 1
    values.sort()
    print(f"两条毫不相干的记录、各 {days} 天：|相关| > 0.5 的占 {big / trials * 100:.1f}%，一半的情况会算出 {values[trials // 2]:.2f} 以上")


两条毫不相干的记录、各 8 天：|相关| > 0.5 的占 21.0%，一半的情况会算出 0.28 以上
两条毫不相干的记录、各 30 天：|相关| > 0.5 的占 0.6%，一半的情况会算出 0.13 以上
两条毫不相干的记录、各 100 天：|相关| > 0.5 的占 0.0%，一半的情况会算出 0.07 以上


**八天数据，两件事明明毫不相干，也有五分之一的一遍会算出 0.5 以上的相关。**

这不是理论把戏。我自己随手记了两列十天的数：通勤 34/41/29/38/45/31/37/40/26/33，午餐花费 32/28/45/30/52/26/38/41/29/35。两列是我分别写的，没有任何共同原因。跑 `corr` 得到 **0.406**——想亲眼验证，可以跑 `corr([34, 41, 29, 38, 45, 31, 37, 40, 26, 33], [32, 28, 45, 30, 52, 26, 38, 41, 29, 35])`。

要是我在周报里写"通勤时间与午餐花费相关 0.41，说明上班路上久了吃得也多"，你没法反驳我——数据就在下面。但上面那一轮实验说明，这个 0.41 完全可能是八天、十天的运气。

所以从今天起，"我看了几天数据，发现 A 和 B 相关"这句话，要自动补上一句：**"我看了几天？"**


## 六、今晚就能用，还有三件事必须说清

代码在 [code/lesson11_correlation_and_basket.py](../code/lesson11_correlation_and_basket.py)。改成你自己的两列记录只需要一步：把 `corr(你的第一列, 你的第二列)` 写出来跑。

**判断你的副业是不是真副业。** 两份收入等权时，合计起伏 = $\sqrt{\rho + (1-\rho)/2}$ 倍的一份的起伏。下面给你三个数（$\rho = 0.7$ 就是都靠同一家平台、同一个行业）：


In [9]:
print("两份收入等权（n = 2）时，合计起伏是多少倍的一份收入：")
for r in [0.7, 0.3, 0.1]:
    print(f"相关 {r:.1f} → {math.sqrt(r + (1 - r) / 2):.3f}")


两份收入等权（n = 2）时，合计起伏是多少倍的一份收入：
相关 0.7 → 0.922
相关 0.3 → 0.806
相关 0.1 → 0.742


翻译成人话：**两份高度相关的收入，跟一份收入差不多（只降了 8%）；换一个不相关的赛道，同样两份钱能降掉四分之一。**

**判断你的账号矩阵是几个账号还是一个账号。** 把三个账号同题的日播放当成 $\rho$ 高的一篮子里的几只，第四节那张地板表直接给出答案。

**在别人的结论里插一句话。** 有人给你看一张散点图或者一个"相关性 0.4"时，问两件事：**几天数据？中间是不是有个共同原因？**

三件事必须说清楚：

**第一，相关不是因果。** $\rho$ 只说"一起动"，不说"谁让谁动"。冰淇淋销量和溺水人数高度相关，共同原因是夏天。第四节那个 `common` 就是这个角色：真正要防的从来不是两件事相关，而是它们背后那个共享的东西（一个平台、一个行业、一个客户、一段宏观）。

**第二，$\rho$ 会在你最需要分散的时候往上跳。** 真实的资产相关在市场大跌时会集体趋近 1——平时各动各的，出事那天全绿。你按平时的 $\rho = 0.3$ 算的地板，出事那天可能是 $\rho = 0.8$ 的地板（$\sqrt{0.8} \approx 0.89$，几乎等于没分散）。这正是第 09 课那个毛病的另一种形态：**你的模型给不了你没放进参数里的事**。

**第三，`corr` 只抓"一起高一起低"这一种关系。** 两件事可以是强关联但非直线（比如收入越高储蓄率越高但会饱和），这时 $\rho$ 会低估它们的关系。反过来说也成立：$\rho$ 接近 0 不代表独立，第 15 课的布朗运动里就有"每天不相关但长期趋势明确"的例子。


## 这一课带走的一句话

起伏可以摊薄，共享的那一部分不能；判断分散别看有几个，看那个消不掉的 $\rho$ 有多大——而且要用够多的天数去量它。

配套课文：[第 11 课（网页版）](../docs/m0-foundations/11-correlation-and-baskets.md)


---

**许可**：正文 CC BY-NC-SA 4.0，代码 MIT，见 [LICENSE.md](../LICENSE.md)。

**建议购买正版**：Paul Glasserman, *Monte Carlo Methods in Financial Engineering*, Springer, 2003；中译本《金融工程中的蒙特卡罗方法》。本 notebook 未复制原书任何文字、图表或表格。
